# Task 1: BM25 baseline

Lexical retrieval over the SciFact abstracts. BM25 is written by hand here (sparse matrix + numpy), tuned on the val split, and the eval/val run files are written at the end.

Runs on CPU in ~2 min.

In [1]:
# only extra package needed here (snowball stemmer)
%pip install -q PyStemmer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 747.7/747.7 kB 12.2 MB/s eta 0:00:00


In [3]:
import os, zipfile
from pathlib import Path

if not list(Path(".").rglob("corpus.jsonl")):
    from google.colab import files
    up = files.upload()                      # pick NLP_Inter_IIT.zip
    zipfile.ZipFile(next(iter(up))).extractall(".")

os.chdir(next(Path(".").rglob("corpus.jsonl")).parent.parent)   # folder that contains data/
print(Path.cwd())

Saving NLP_Inter_IIT.zip to NLP_Inter_IIT.zip
/content/NLP_Inter_IIT


In [4]:
import json, math, re, sys, time, itertools
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import Stemmer

# folder layout:
#   NLP_Inter_IIT/
#     data/  check_format.py  results_template.json
#     code/         <- this notebook
#     predictions/  <- run files for the submission (created below)
#     work/         <- cached scores that the later notebooks reuse (not submitted)
#
# on colab: upload NLP_Inter_IIT to drive, then uncomment these 3 lines
# from google.colab import drive; drive.mount("/content/drive")
# import os; os.chdir("/content/drive/MyDrive/NLP_Inter_IIT")
# (the pip cell above has to be run again after a runtime restart)

ROOT = Path.cwd()
if not (ROOT / "data" / "corpus.jsonl").exists():
    ROOT = ROOT.parent          # notebook is inside code/
assert (ROOT / "data" / "corpus.jsonl").exists(), "open this notebook from NLP_Inter_IIT/ or NLP_Inter_IIT/code/"

DATA = ROOT / "data"
PRED = ROOT / "predictions"
WORK = ROOT / "work"
PRED.mkdir(exist_ok=True)
WORK.mkdir(exist_ok=True)

SEED = 42
np.random.seed(SEED)
print(ROOT)

/content/NLP_Inter_IIT


## Data

In [5]:
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

# sort corpus by doc_id and claims by id, so every score matrix saved later
# has the same row/column order in all notebooks
corpus = sorted(read_jsonl(DATA / "corpus.jsonl"), key=lambda d: d["doc_id"])
doc_ids = np.array([d["doc_id"] for d in corpus])

claims = {}
for split in ["train", "val", "eval"]:
    claims[split] = sorted(read_jsonl(DATA / f"{split}_claims.jsonl"), key=lambda c: c["id"])

print(len(corpus), "abstracts")
print({s: len(c) for s, c in claims.items()})

5183 abstracts
{'train': 647, 'val': 162, 'eval': 300}


In [6]:
# quick look at the data
n_words = pd.Series([len((d["title"] + " " + " ".join(d["abstract"])).split()) for d in corpus])
print("words per abstract (title included):")
print(n_words.describe().round(1).to_string())

for split in ["train", "val"]:
    cs = claims[split]
    n_docs = Counter(len(c["evidence"]) for c in cs)
    labels = Counter(e["label"] for c in cs for e in c["evidence"])
    print(f"\n{split}: {len(cs)} claims, {n_docs[0]} have no evidence (NEI)")
    print("  gold abstracts per claim:", dict(sorted(n_docs.items())))
    print("  labels:", dict(labels))

words per abstract (title included):
count    5183.0
mean      214.6
std        87.3
min        33.0
25%       158.0
50%       204.0
75%       260.0
max      1541.0

train: 647 claims, 248 have no evidence (NEI)
  gold abstracts per claim: {0: 248, 1: 374, 2: 14, 3: 6, 4: 5}
  labels: {'CONTRADICT': 150, 'SUPPORT': 290}

val: 162 claims, 56 have no evidence (NEI)
  gold abstracts per claim: {0: 56, 1: 94, 2: 6, 3: 6}
  labels: {'SUPPORT': 80, 'CONTRADICT': 44}


In [7]:
# one example
c = claims["val"][4]
ev = c["evidence"][0]
doc = corpus[list(doc_ids).index(ev["doc_id"])]
print("claim   :", c["claim"])
print("label   :", ev["label"])
print("abstract:", doc["title"])
for i in ev["sentences"]:
    print("evidence:", doc["abstract"][i])

claim   : A deficiency of folate increases blood levels of homocysteine.
label   : SUPPORT
abstract: Randomized trial of folic acid supplementation and serum homocysteine levels.
evidence: RESULTS Median serum homocysteine level decreased with increasing folic acid dosage, to a maximum at 0.8 mg of folic acid per day, when the homocysteine reduction (placebo adjusted) was 2.7 micromol/L (23%), similar to the known effect of folic acid dosages of 1 mg/d and above.


Things to note:
- about 1/3 of the claims have no evidence at all. Retrieval metrics skip those, but the eval file still needs all 300 claims.
- most claims with evidence have exactly one gold abstract, so nDCG@10 is mostly about how high that one abstract is ranked.
- abstracts are ~215 words on average, a few are very long (relevant later for max_length of the dense models).

## Metrics and run file helpers

In [8]:
TOP_K = 100

def gold_docs(split):
    # claim id -> set of gold doc ids (empty set for NEI claims)
    return {c["id"]: {e["doc_id"] for e in c["evidence"]} for c in claims[split]}

def make_run(scores, split, k=TOP_K):
    # scores: (n_claims, n_docs) matrix -> {claim_id: [(doc_id, score), ...]} with the top k docs
    # stable sort so that ties always come out in the same order
    order = np.argsort(-scores, axis=1, kind="stable")[:, :k]
    run = {}
    for i, c in enumerate(claims[split]):
        run[c["id"]] = [(int(doc_ids[j]), float(scores[i, j])) for j in order[i]]
    return run

def claim_metrics(run, gold):
    # per-claim nDCG@10, recall@100 and reciprocal rank (binary relevance)
    rows = {}
    for cid, g in gold.items():
        if len(g) == 0:
            continue                      # NEI claims are not scored
        ranked = [d for d, _ in run[cid]]
        dcg = sum(1 / math.log2(r + 2) for r, d in enumerate(ranked[:10]) if d in g)
        idcg = sum(1 / math.log2(r + 2) for r in range(min(len(g), 10)))
        rr = 0.0
        for r, d in enumerate(ranked):
            if d in g:
                rr = 1 / (r + 1)
                break
        rows[cid] = {"ndcg10": dcg / idcg, "recall100": len(g & set(ranked)) / len(g), "mrr": rr}
    return pd.DataFrame.from_dict(rows, orient="index")

def evaluate(run, gold):
    return claim_metrics(run, gold).mean().round(4).to_dict()

def write_trec(path, run, name):
    # claim_id Q0 doc_id rank score run_name
    with open(path, "w") as f:
        for cid in sorted(run):
            for rank, (did, score) in enumerate(run[cid], start=1):
                f.write(f"{cid} Q0 {did} {rank} {score:.6f} {name}\n")

## BM25

$$\text{score}(q,d)=\sum_{t\in q} \text{idf}(t)\cdot\frac{tf_{t,d}\,(k_1+1)}{tf_{t,d}+k_1\,(1-b+b\,|d|/\text{avgdl})}$$

with $\text{idf}(t)=\log\big(1+\frac{N-df_t+0.5}{df_t+0.5}\big)$ (the Lucene version, which can't go negative).

`k1` = how quickly repeated terms saturate, `b` = strength of length normalisation.

In [9]:
stemmer = Stemmer.Stemmer("english")

STOPWORDS = set("""a about above after again against all am an and any are as at be because been before
being below between both but by can did do does doing down during each few for from further had has have
having he her here hers him his how i if in into is it its itself just me more most my no nor not of off
on once only or other our ours out over own same she should so some such than that the their theirs them
then there these they this those through to too under until up very was we were what when where which
while who whom why will with you your yours""".split())

def tokenize(text, stem=True, stop=True):
    toks = re.findall(r"[a-z0-9]+", text.lower())
    if stop:
        toks = [t for t in toks if t not in STOPWORDS]
    if stem:
        toks = stemmer.stemWords(toks)
    return toks

tokenize("Drug X reduces the risk of myocardial infarctions in COVID-19 patients.")

['drug', 'x', 'reduc', 'risk', 'myocardi', 'infarct', 'covid', '19', 'patient']

In [10]:
class BM25:
    # tf is stored as a sparse (n_docs x vocab) matrix. For a given (k1, b) it is converted
    # into a matrix of bm25 weights, and scoring all queries is then a single sparse product.

    def __init__(self, docs, stem=True, stop=True, title_weight=1):
        self.stem = stem
        self.stop = stop
        self.vocab = {}
        rows, cols, vals = [], [], []
        for i, d in enumerate(docs):
            # repeating the title tokens = simple title boost
            toks = tokenize(d["title"], stem, stop) * title_weight
            toks += tokenize(" ".join(d["abstract"]), stem, stop)
            for t, tf in Counter(toks).items():
                if t not in self.vocab:
                    self.vocab[t] = len(self.vocab)
                rows.append(i)
                cols.append(self.vocab[t])
                vals.append(tf)

        N = len(docs)
        self.tf = sp.csr_matrix((vals, (rows, cols)), shape=(N, len(self.vocab)), dtype=np.float32)
        self.doc_len = np.asarray(self.tf.sum(axis=1)).ravel()
        self.avgdl = self.doc_len.mean()
        df = np.asarray((self.tf > 0).sum(axis=0)).ravel()
        self.idf = np.log(1 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)
        # row index of every non-zero entry, needed for the per-document length norm
        self.nz_rows = np.repeat(np.arange(N), np.diff(self.tf.indptr))

    def weights(self, k1, b):
        norm = k1 * (1 - b + b * self.doc_len / self.avgdl)
        w = self.tf.copy()
        w.data = w.data * (k1 + 1) / (w.data + norm[self.nz_rows])
        return w @ sp.diags(self.idf)

    def encode_queries(self, texts):
        rows, cols, vals = [], [], []
        for i, text in enumerate(texts):
            for t, qtf in Counter(tokenize(text, self.stem, self.stop)).items():
                if t in self.vocab:       # terms not in the corpus can't match anything
                    rows.append(i)
                    cols.append(self.vocab[t])
                    vals.append(qtf)
        return sp.csr_matrix((vals, (rows, cols)), shape=(len(texts), len(self.vocab)), dtype=np.float32)

    def score(self, texts, k1=1.2, b=0.75):
        q = self.encode_queries(texts)
        return np.asarray((q @ self.weights(k1, b).T).todense(), dtype=np.float32)

In [11]:
# baseline with the usual defaults, no stemming / stopword removal / title boost
val_queries = [c["claim"] for c in claims["val"]]
gold_val = gold_docs("val")

bm25 = BM25(corpus, stem=False, stop=False, title_weight=1)
print("vocab:", len(bm25.vocab), " avg doc len:", round(bm25.avgdl, 1))

default_val = evaluate(make_run(bm25.score(val_queries, k1=1.2, b=0.75), "val"), gold_val)
default_val

vocab: 35734  avg doc len: 225.2


{'ndcg10': 0.8244, 'recall100': 0.9906, 'mrr': 0.7975}

## Tuning on val

Grid over tokenisation (stemming, stopwords), title weight, k1 and b. As per the rules the best setting is the one with the highest val nDCG@10.

In [12]:
grid_k1 = [0.6, 0.9, 1.2, 1.5, 2.0]
grid_b = [0.3, 0.5, 0.65, 0.75, 0.9, 1.0]

rows = []
t0 = time.time()
for stem, stop, tw in itertools.product([False, True], [False, True], [1, 2, 3, 4, 5]):
    bm25 = BM25(corpus, stem, stop, tw)       # the index only depends on these three
    for k1, b in itertools.product(grid_k1, grid_b):
        m = evaluate(make_run(bm25.score(val_queries, k1, b), "val"), gold_val)
        rows.append({"stem": stem, "stop": stop, "title_weight": tw, "k1": k1, "b": b, **m})
print(f"{len(rows)} settings, {time.time() - t0:.0f}s")

grid = pd.DataFrame(rows).sort_values(["ndcg10", "recall100"], ascending=False).reset_index(drop=True)
grid.head(10)

600 settings, 77s


,stem,stop,title_weight,k1,b,ndcg10,recall100,mrr
0,True,False,5,0.9,1.00,0.8735,0.9811,0.8546
1,True,False,3,2.0,0.65,0.8732,0.9811,0.8563
2,True,False,5,0.9,0.90,0.8732,0.9811,0.8556
3,True,False,4,1.5,0.50,0.8726,0.9811,0.8538
4,True,True,5,0.9,1.00,0.8723,0.9906,0.8531
5,True,False,3,1.5,0.65,0.8722,0.9811,0.8550
6,True,False,3,0.9,0.90,0.8720,0.9811,0.8493
7,True,False,3,1.2,0.65,0.8717,0.9811,0.8527
8,True,True,4,0.9,0.90,0.8716,0.9906,0.8485
9,True,False,3,2.0,0.50,0.8713,0.9811,0.8517


In [13]:
# best val nDCG@10 reachable for each value of each parameter
for col in ["stem", "stop", "title_weight", "k1", "b"]:
    print(grid.groupby(col)["ndcg10"].max().round(4).to_dict(), " <-", col)

{False: 0.8481, True: 0.8735}  <- stem
{False: 0.8735, True: 0.8723}  <- stop
{1: 0.8637, 2: 0.869, 3: 0.8732, 4: 0.8726, 5: 0.8735}  <- title_weight
{0.6: 0.8674, 0.9: 0.8735, 1.2: 0.8717, 1.5: 0.8726, 2.0: 0.8732}  <- k1
{0.3: 0.871, 0.5: 0.8726, 0.65: 0.8732, 0.75: 0.8708, 0.9: 0.8732, 1.0: 0.8735}  <- b


Stemming is the only thing that clearly matters (+0.025). The title boost gives about +0.01. k1 and b hardly change anything: the top settings are within 0.002 of each other, which on 106 scored val claims is noise. Taking the top row since that's the selection rule.

In [14]:
best = grid.iloc[0]
params = {"stem": bool(best["stem"]), "stop": bool(best["stop"]), "title_weight": int(best["title_weight"]),
          "k1": float(best["k1"]), "b": float(best["b"])}
params

{'stem': True, 'stop': False, 'title_weight': 5, 'k1': 0.9, 'b': 1.0}

## Final run files

In [15]:
bm25 = BM25(corpus, params["stem"], params["stop"], params["title_weight"])

metrics = {}
for split in ["train", "val", "eval"]:
    scores = bm25.score([c["claim"] for c in claims[split]], params["k1"], params["b"])
    # full score matrices are kept for the hybrid (task 3) and for hard negative mining (task 4)
    np.save(WORK / f"scores_bm25_{split}.npy", scores)
    run = make_run(scores, split)
    if split != "eval":
        metrics[split] = evaluate(run, gold_docs(split))
        print(split, metrics[split])
    if split == "val":
        write_trec(PRED / "val_bm25.trec", run, "bm25")
    if split == "eval":
        write_trec(PRED / "t1_bm25.trec", run, "bm25")

train {'ndcg10': 0.8519, 'recall100': 0.9881, 'mrr': 0.831}
val {'ndcg10': 0.8735, 'recall100': 0.9811, 'mrr': 0.8546}


Train nDCG@10 is close to val, so the tuned settings are not just fitted to the val split.

In [16]:
# sanity check with the official checker
sys.path.append(str(ROOT))
from check_format import check_trec

all_docs = set(doc_ids.tolist())
check_trec(PRED / "t1_bm25.trec", {c["id"] for c in claims["eval"]}, all_docs)
check_trec(PRED / "val_bm25.trec", {c["id"] for c in claims["val"]}, all_docs)
print("both run files pass check_format")

both run files pass check_format


In [17]:
# keep the settings + val numbers, they go into results.json at the end
settings = (f"own implementation, Lucene idf, k1={params['k1']}, b={params['b']}, "
            f"{'snowball stemming' if params['stem'] else 'no stemming'}, "
            f"{'stopwords removed' if params['stop'] else 'stopwords kept'}, "
            f"title x{params['title_weight']}")

log_path = WORK / "results_log.json"
results_log = json.load(open(log_path)) if log_path.exists() else {}
results_log["task1"] = {"bm25_settings": settings, "params": params,
                        "val": metrics["val"], "train": metrics["train"],
                        "val_ndcg10_default": default_val["ndcg10"]}
json.dump(results_log, open(log_path, "w"), indent=2)
print(settings)
print("val nDCG@10:", metrics["val"]["ndcg10"])

own implementation, Lucene idf, k1=0.9, b=1.0, snowball stemming, stopwords kept, title x5
val nDCG@10: 0.8735


## Where BM25 fails

Val claims where no gold abstract made it into the top 10.

In [18]:
val_run = make_run(np.load(WORK / "scores_bm25_val.npy"), "val")
per_claim = claim_metrics(val_run, gold_val)
missed = per_claim[per_claim["ndcg10"] == 0].index.tolist()
print(len(missed), "of", len(per_claim), "claims\n")

docs_by_id = {d["doc_id"]: d for d in corpus}
claim_text = {c["id"]: c["claim"] for c in claims["val"]}

for cid in missed:
    gold_id = sorted(gold_val[cid])[0]
    doc = docs_by_id[gold_id]
    ranked = [d for d, _ in val_run[cid]]
    rank = ranked.index(gold_id) + 1 if gold_id in ranked else ">100"
    shared = set(tokenize(claim_text[cid])) & set(tokenize(doc["title"] + " " + " ".join(doc["abstract"])))
    print(f"[{cid}] {claim_text[cid]}")
    print(f"    gold at rank {rank}: {doc['title']}")
    print(f"    shared terms: {sorted(shared)}\n")

5 of 106 claims

[88] Adult tissue-resident macrophages possess a self-renewing capacity.
    gold at rank 24: Fate mapping reveals origins and dynamics of monocytes and tissue macrophages under homeostasis.
    shared terms: ['macrophag', 'resid', 'tissu']

[382] Environmentally-induced senescence is mimicked in disease modeling by adding specific molecules to the culture system.
    gold at rank >100: Human iPSC-based modeling of late-onset disease via progerin-induced aging.
    shared terms: ['diseas', 'induc', 'model', 'specif']

[430] FoxO3a activation in neuronal death is inhibited by reactive oxygen species (ROS).
    gold at rank 96: Stress-dependent regulation of FOXO transcription factors by the SIRT1 deacetylase.
    shared terms: ['death', 'inhibit', 'speci']

[463] Genomic aberrations of metastases provide information for targeted therapy.
    gold at rank 13: Evolution of metastasis revealed by mutational landscapes of chemically induced skin cancers
    shared terms: ['

In [19]:
import zipfile
zip_path = Path.cwd() / "NLP_Inter_IIT.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for item in ["data", "predictions", "work", "check_format.py", "results_template.json"]:
        p = ROOT / item
        if not p.exists():
            continue
        for f in ([p] if p.is_file() else sorted(p.rglob("*"))):
            if f.is_file():
                z.write(f, Path("NLP_Inter_IIT") / f.relative_to(ROOT))
from google.colab import files
files.download(str(zip_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In these cases the claim is a paraphrase of the abstract (e.g. "geographically segregated" vs "spatially organized"), so the only overlap is on generic words like *disease*, *model*, *response*. This is the gap the dense models in Task 2 should close.